# 🎭 Cheeni Capability 3: Speech Emotion & Mood Recognition (SER)
### Fine-Tuning `facebook/wav2vec2-base` on RAVDESS Emotional Speech Corpus

**Core Capability**: Real-time vocal mood detection (Neutral, Calm, Happy, Sad, Angry, Fearful, Disgust, Surprised) to dynamically adjust Cheeni's personality and tone.
**Benchmark Dataset**: Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS, 1,440 validated clips across 24 professional actors) loaded via `xbgoose/ravdess` (Parquet).
**Target Hugging Face Hub**: `wraithklu/cheeni-emotion-vad_03`
**Compute Environment**: Google Colab (Tesla T4 GPU or A100)

In [ ]:
# 1. Install Audio Processing & Hugging Face Libraries
!pip install -q transformers datasets evaluate accelerate soundfile librosa huggingface_hub

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
login()

In [ ]:
# 3. Stream RAVDESS Emotion Benchmark via Direct Parquet
from datasets import load_dataset, Audio

print("⚡ Streaming RAVDESS Speech Emotion dataset directly into Colab...")
raw_dataset = load_dataset("xbgoose/ravdess", split="train")

# Ensure standard 16kHz audio sampling rate
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))
print(f"✅ Total authentic emotional speech recordings: {len(raw_dataset)}")
print(f"First sample emotion: {raw_dataset[0]['emotion']}")

In [ ]:
# 4. Define 8 Emotion Classes and Bidirectional Mappings
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
label2id = {emotion: i for i, emotion in enumerate(EMOTIONS)}
id2label = {i: emotion for i, emotion in enumerate(EMOTIONS)}

print("Configured 8 Emotion Classes:")
for k, v in id2label.items():
    print(f"  [{k}] -> {v}")

In [ ]:
# 5. Preprocess Audio Waveforms with Feature Extractor
from transformers import AutoFeatureExtractor

MODEL_NAME = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def preprocess_function(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        max_length=16000 * 4,  # Exactly 64,000 samples (4.0s)
        truncation=True,
        padding="max_length",
    )
    inputs["labels"] = [label2id[emotion.lower()] for emotion in batch["emotion"]]
    return inputs

print("Processing acoustic features (wav2vec2 16kHz tensors)...")
encoded_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=[col for col in raw_dataset.column_names if col != "labels"]
)

# 80/20 Stratified train/validation split
split = encoded_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(eval_dataset)}")

In [ ]:
# 6. Metrics Computation (Accuracy + Macro F1)
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    references = eval_pred.label_ids
    acc = accuracy_metric.compute(predictions=predictions, references=references)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=references, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

In [ ]:
# 7. Load Audio Classification Model Architecture
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(EMOTIONS),
    label2id=label2id,
    id2label=id2label,
)

# Freeze CNN feature extraction layers to preserve low-level acoustic representations
model.freeze_feature_encoder()
print("✅ wav2vec2 model loaded with 8 emotion classes and frozen CNN feature extractor")

In [ ]:
# 8. Train Model on GPU with Evaluation Every Epoch
training_args = TrainingArguments(
    output_dir="./cheeni-emotion-vad-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Training Speech Emotion Model on GPU...")
trainer.train()

In [ ]:
# 9. Test Live Emotion Inference on Validation Audio
import torch

model.eval()
test_sample = eval_dataset[0]["input_values"]
true_label_idx = eval_dataset[0]["labels"]
true_emotion = id2label[true_label_idx]

test_input = torch.tensor([test_sample]).to(model.device).to(model.dtype)

with torch.no_grad():
    logits = model(test_input).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred_idx = torch.argmax(probs).item()
    pred_emotion = id2label[pred_idx]
    confidence = probs[pred_idx].item()

print(f"🎯 Predicted Emotion: {pred_emotion.upper()} (Confidence: {confidence:.2%})")
print(f"🏷️ True Ground Truth: {true_emotion.upper()}")
print("\nAll Emotion Probabilities:")
for i, emo in id2label.items():
    print(f"  {emo:10s}: {probs[i].item():.2%}")

In [ ]:
# 10. Push Fine-Tuned Emotion Model Directly to Hugging Face Hub
HF_REPO_ID = "wraithklu/cheeni-emotion-vad_03"

print(f"Pushing trained Emotion model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID)
feature_extractor.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")